# A two-joint arm, on one wire

The simulated machine is a BUS of joints: `LA` is the left arm segment,
unit 1 the shoulder, unit 2 the elbow - each a whole board with its own
drive and shaft sensor, exactly how a real arm daisy-chains this
controller over RS485. Two `Coaxial63100` sessions, two `motion.servo`
blocks, one reach.

The servo corrects between moves (see `position_servo.ipynb` for why),
which is an ARM's cadence anyway: plan, move, settle, verify.

In [ ]:
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd()
while not (root / 'host' / 'coaxial').is_dir():
    root = root.parent
sys.path.insert(0, str(root / 'host'))

from coaxial import Coaxial63100

SIMULATED = True                    # False at the bench
BUS = 'LA'                # the left arm's RS485 segment

joints = {}
for name, unit in (('shoulder', 1), ('elbow', 2)):
    rig = Coaxial63100(port=BUS, unit=unit, simulated_device=SIMULATED,
                       power_afe=False).open()
    rig.drive.source('model')
    rig.gates.arm(bypass_sto=True, ignore_interlock=True)
    joints[name] = rig
    print(name, rig)
BLUE, RED, GREY = '#1f4e79', '#c0392b', '#95a5a6'
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.15})


In [ ]:
L1, L2 = 0.30, 0.25       # m, shoulder and forearm links


def tip(q1, q2):
    """Forward kinematics, planar."""
    a1, a2 = math.radians(q1), math.radians(q1 + q2)
    return (L1 * math.cos(a1) + L2 * math.cos(a2),
            L1 * math.sin(a1) + L2 * math.sin(a2))


WAYPOINTS = [(0.0, 0.0), (35.0, 40.0), (55.0, 15.0), (20.0, 70.0),
             (0.0, 0.0)]

path = []
with joints['shoulder'].motion.servo(amps=3.0) as shoulder, \
     joints['elbow'].motion.servo(amps=3.0) as elbow:
    for q1, q2 in WAYPOINTS:
        got1 = shoulder.to(q1, tol=0.8)
        got2 = elbow.to(q2, tol=0.8)
        path.append((got1, got2))
        print('asked (%5.1f, %5.1f)  joints (%5.1f, %5.1f)  tip (%.3f, %.3f) m'
              % (q1, q2, got1, got2, *tip(got1, got2)))

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5))
for k, (q1, q2) in enumerate(path):
    a1, a2 = math.radians(q1), math.radians(q1 + q2)
    xs = [0, L1 * math.cos(a1), L1 * math.cos(a1) + L2 * math.cos(a2)]
    ys = [0, L1 * math.sin(a1), L1 * math.sin(a1) + L2 * math.sin(a2)]
    ax.plot(xs, ys, 'o-', color=BLUE, alpha=0.25 + 0.75 * k / len(path))
tips = [tip(*p) for p in path]
ax.plot(*zip(*tips), color=RED, lw=1.2, label='tip path')
ax.set(xlabel='m', ylabel='m', title='the reach, joint by joint',
       aspect='equal')
ax.legend(frameon=False)
fig.tight_layout()
for rig in joints.values():
    rig.close()

## What scales

A joint is a unit id: six axes is six ids on the segment, addressed one
`Coaxial63100` each over the shared broker. What this arm does NOT have
yet is coordination - each joint settles alone, in sequence. Simultaneous
trajectories want interleaved `_slew` passes on one scheduler, and
gravity feedforward wants the identified masses; both are host code on
the verbs above, no firmware between them and a working cell.